# La transition vers la voiture électrique en France

Ouvrir ce notebook à la racine du dépôt. Le classeur AAV est nécessaire pour construire le référentiel communal. Les autres fichiers sont listés dans le README.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from src import config, preparation, fusion, indicateurs, graphiques

disponible = {nom: preparation.chemin_brut(nom).is_file() for nom in config.FICHIERS}
display(pd.DataFrame([
    {"source": nom, "fichier": fichier, "présent": disponible[nom]}
    for nom, fichier in config.FICHIERS.items()
]))
if not disponible["aav"]:
    raise FileNotFoundError(f"Classeur AAV manquant : {preparation.chemin_brut('aav')}")


,source,fichier,présent
0,parc,Donnees-sur-le-parc-de-vehicules-au-niveau-com...,False
1,neuf,Immatriculations-de-vehicules-achetes-neufs-au...,False
2,occasion,Immatriculations-de-vehicules-achetes-doccasio...,False
3,aav,AAV2020_au_01-01-2026.xlsx,True
4,filosofi,DS_FILOSOFI_CC_2023_data.csv,False
5,irve,consolidation-etalab-schema-irve-statique-v-2_...,False
6,bpe,DS_BPE_2025_data.csv,False
7,population,DS_POPULATIONS_REFERENCE_2023_data.csv,False
8,contours,communes-50m.geojson,False


## 1. Référentiel communal

In [2]:
referentiel = preparation.preparer_aav()
print(f"{len(referentiel):,} communes dans le référentiel.")


34,746 communes dans le référentiel.


## 2. Préparation des sources

In [3]:
tables = {}

for nom in ("parc", "neuf", "occasion"):
    if disponible[nom]:
        est_parc = nom == "parc"
        colonne = f"PARC_{config.ANNEE_PARC}" if est_parc else f"IMMAT_{config.ANNEE_IMMAT}"
        brut = preparation.lire_fichier_sdes(nom, [colonne])
        tables[nom] = preparation.preparer_vehicules(
            brut, colonne,
            config.ELEC_PARC if est_parc else config.ELEC_IMMAT,
            config.MENAGE_PARC if est_parc else config.MENAGE_IMMAT,
            nom,
        )

for nom, preparer in (
    ("filosofi", preparation.preparer_filosofi),
    ("population", preparation.preparer_population),
    ("bpe", preparation.preparer_bpe),
):
    if disponible[nom]:
        tables[nom] = preparer()

if disponible["irve"]:
    tables["irve"], points = preparation.preparer_irve(referentiel["code_commune"])
    display(points["rattachement"].value_counts())


## 3. Jointures et couverture des sources

In [4]:
if tables:
    base, couverture = fusion.construire_base(referentiel, tables)
    display(couverture)
else:
    base = referentiel.copy()
    print("Aucune source à joindre au référentiel.")


Aucune source à joindre au référentiel.


## 4. Indicateurs et contrôles

In [5]:
colonnes_sources = [
    f"vp_{nom}{suffixe}"
    for nom in ("parc", "neuf", "occasion")
    for suffixe in ("", "_elec", "_men", "_elec_men")
] + [
    "niveau_vie_median", "population", "nb_points", "nb_points_rapides",
    "nb_gares", "nb_equip_transport",
]

for colonne in colonnes_sources:
    if colonne not in base:
        base[colonne] = float("nan")

base = indicateurs.calculer_indicateurs(
    base, irve_exhaustif=disponible["irve"], bpe_exhaustive=disponible["bpe"]
)
base["gare"] = base["gare"].where(base["nb_gares"].notna())
base, bornes_revenu = indicateurs.ajouter_groupes_revenu(base)
base = indicateurs.ajouter_classes_recharge(base)
display(indicateurs.controler_base(base))


Pas de niveau de vie communal : groupes non calculés.


,effectifs négatifs,électriques > total,"parts hors [0, 100]",parts indéfinies
table,,,,
parc,0,0,0,34746
neuf,0,0,0,34746
occasion,0,0,0,34746


## 5. Tableaux et graphiques

In [6]:
display(indicateurs.tableau_synthese(base))
if bornes_revenu is not None:
    display(bornes_revenu)

graphiques.appliquer_style()
for groupe, libelle in (
    ("position_aav", "Position dans l’aire d’attraction"),
    ("groupe_revenu", "Niveau de vie"),
    ("classe_recharge", "Offre de recharge"),
):
    if groupe not in base:
        continue
    resume = indicateurs.resume_par_groupe(base, groupe)
    if not resume.empty:
        display(resume)
        graphiques.tracer_barres(
            resume, f"Part électrique du parc · {libelle}",
            f"parc_{groupe}", label_groupe=libelle, trier=False,
        )
        plt.show()

if disponible["contours"] and base["part_parc"].notna().any():
    graphiques.tracer_carte(base, "part_parc", "Part électrique du parc", "carte_communes")
    plt.show()


,nb_communes
region,
Auvergne-Rhône-Alpes,4025
Bourgogne-Franche-Comté,3685
Bretagne,1202
Centre-Val de Loire,1754
Corse,360
Grand Est,5115
Hauts-de-France,3782
Normandie,2644
Nouvelle-Aquitaine,4293


## 6. Export de la base communale

In [7]:
config.DOSSIER_PREP.mkdir(parents=True, exist_ok=True)
sortie = config.DOSSIER_PREP / "base_communale.csv"
base.to_csv(sortie, index=False)
print(sortie.relative_to(config.RACINE))


data\processed\base_communale.csv
